# Phân tách nguồn âm thanh (Music Source Separation) với DemucsLite trên máy cục bộ

Notebook này được cấu hình chạy trực tiếp trên laptop/máy trạm cá nhân, sử dụng thư mục dữ liệu `musdb18hq` đã giải nén sẵn.

## 1. Thiết lập thư mục và cấu trúc dự án

In [12]:
import os
import imageio_ffmpeg

# Lấy đường dẫn ffmpeg và thêm thư mục chứa nó vào PATH hệ thống
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_dir = os.path.dirname(ffmpeg_exe)
os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ["PATH"]

In [13]:
import os
from pathlib import Path

# Thư mục dự án là thư mục hiện tại chứa notebook
PROJECT_ROOT = Path(".").resolve()

# Đường dẫn tới thư mục musdb18hq đã giải nén trên máy của bạn
# Lưu ý: Thay đổi đường dẫn nếu bạn đặt ở ổ đĩa khác (ví dụ: Path("D:/musdb18hq"))
DATA_ROOT = PROJECT_ROOT / "musdb18hq"

DATA_DIR = PROJECT_ROOT / "data"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATA_ROOT} (Tồn tại: {DATA_ROOT.exists()})")

Project root: D:\TDTU\HK7\Tài tử gặp Phong lưu
Dataset root: D:\TDTU\HK7\Tài tử gặp Phong lưu\musdb18hq (Tồn tại: True)


## 2. Kiến trúc mô hình DemucsLite (Giữ nguyên gốc)

In [14]:
%%writefile scripts/demucs_lite.py
"""
Kiến trúc mạng U-Net cơ sở dựa trên Demucs (tinh gọn)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, stride)
        self.glu_conv = nn.Conv1d(out_channels, 2 * out_channels, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.glu_conv(x)
        x = F.glu(x, dim=1)
        return x


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, is_last=False):
        super().__init__()
        self.glu_conv = nn.Conv1d(in_channels, 2 * in_channels, kernel_size=3, padding=1)
        self.deconv = nn.ConvTranspose1d(in_channels, out_channels, kernel_size, stride)
        self.is_last = is_last

    def forward(self, x, skip):
        x = x + skip
        x = self.glu_conv(x)
        x = F.glu(x, dim=1)
        x = self.deconv(x)
        if not self.is_last:
            x = F.relu(x)
        return x


class DemucsLite(nn.Module):
    def __init__(
        self,
        sources=4,
        audio_channels=2,
        channels=32,
        depth=5,
        growth=2,
        kernel_size=8,
        stride=4,
        lstm_layers=1,
    ):
        super().__init__()
        self.sources = sources
        self.audio_channels = audio_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.depth = depth

        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()

        in_ch = audio_channels
        out_ch = channels
        channel_sizes = []
        for i in range(depth):
            self.encoders.append(EncoderBlock(in_ch, out_ch, kernel_size, stride))
            channel_sizes.append(out_ch)
            in_ch = out_ch
            out_ch = int(out_ch * growth)

        bottleneck_ch = channel_sizes[-1]
        self.lstm = nn.LSTM(
            input_size=bottleneck_ch,
            hidden_size=bottleneck_ch,
            num_layers=lstm_layers,
            bidirectional=True,
            batch_first=True,
        )
        self.lstm_proj = nn.Linear(2 * bottleneck_ch, bottleneck_ch)

        rev_channels = list(reversed(channel_sizes))
        for i in range(depth):
            in_ch = rev_channels[i]
            if i + 1 < depth:
                out_ch = rev_channels[i + 1]
            else:
                out_ch = sources * audio_channels
            is_last = i == depth - 1
            self.decoders.append(DecoderBlock(in_ch, out_ch, kernel_size, stride, is_last))

    def valid_length(self, length):
        for _ in range(self.depth):
            length = math.ceil((length - self.kernel_size) / self.stride) + 1
            length = max(length, 1)
        for _ in range(self.depth):
            length = (length - 1) * self.stride + self.kernel_size
        return int(length)

    def forward(self, mixture):
        original_length = mixture.shape[-1]
        target_length = self.valid_length(original_length)
        pad_amount = target_length - original_length
        x = F.pad(mixture, (0, pad_amount))

        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)

        x_lstm = x.permute(0, 2, 1)
        x_lstm, _ = self.lstm(x_lstm)
        x_lstm = self.lstm_proj(x_lstm)
        x = x_lstm.permute(0, 2, 1)

        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)

        x = x[..., :original_length]
        batch = x.shape[0]
        x = x.view(batch, self.sources, self.audio_channels, -1)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

Overwriting scripts/demucs_lite.py


## 3. Tạo script quét dataset `scripts/verify_and_split_dataset.py`

In [15]:
%%writefile scripts/verify_and_split_dataset.py
import argparse
import csv
from pathlib import Path
import musdb
import soundfile as sf

EXPECTED_COUNTS = {"train": 100, "test": 50}
EXPECTED_SR = 44100
EXPECTED_CHANNELS = 2

def scan_subset(root: str, subset: str):
    mus = musdb.DB(root=root, is_wav=True, subsets=subset)
    records = []
    for track in mus:
        mix_path = Path(track.path)
        info = sf.info(str(mix_path))
        records.append({
            "track_name": track.name,
            "subset": subset,
            "duration_sec": round(info.duration, 2),
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "path": str(mix_path.parent),
        })
    return records

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", required=True)
    parser.add_argument("--output", default="data/manifest.csv")
    parser.add_argument("--val-ratio", type=float, default=0.15)
    args = parser.parse_args()

    all_records = []
    for subset in ["train", "test"]:
        records = scan_subset(args.root, subset)
        print(f"[{subset.upper()}] Tìm thấy {len(records)} track.")
        all_records.extend(records)

    if args.val_ratio > 0:
        import random
        rng = random.Random(42)
        train_idx = [i for i, r in enumerate(all_records) if r["subset"] == "train"]
        rng.shuffle(train_idx)
        n_val = int(len(train_idx) * args.val_ratio)
        for i in train_idx[:n_val]:
            all_records[i]["subset"] = "val"
        print(f"Đã tách {n_val} track train sang validation.")

    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(all_records[0].keys()))
        writer.writeheader()
        writer.writerows(all_records)
    print(f"Đã lưu manifest tại: {out_path}")

if __name__ == "__main__":
    main()

Overwriting scripts/verify_and_split_dataset.py


## 4. Kiểm tra và phân chia tập train/validation/test

In [16]:
!python scripts/verify_and_split_dataset.py \
    --root "{DATA_ROOT}" \
    --output data/manifest.csv \
    --val-ratio 0.15

[TRAIN] Tìm thấy 100 track.
[TEST] Tìm thấy 50 track.
Đã tách 15 track train sang validation.
Đã lưu manifest tại: data\manifest.csv


In [17]:
import pandas as pd
df = pd.read_csv('data/manifest.csv')
print(df['subset'].value_counts())

subset
train    85
test     50
val      15
Name: count, dtype: int64


## 5. Dataset và DataLoader (Tối ưu tài nguyên Laptop)

In [3]:
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader
import pitch_speed_perturb
 
SEED = 42
SAMPLE_RATE = 44100
SEGMENT_SECONDS = 10
BATCH_SIZE = 4
TRAIN_CROPS_PER_TRACK = 8
EVAL_CROPS_PER_TRACK = 8
NUM_WORKERS = 0
TARGET_SOURCES = ("vocals", "drums", "bass", "other")
REPORT_SOURCES = ("vocals", "drums", "bass")
 
# Xác suất áp dụng pitch/speed perturbation cho 1 sample (chỉ khi augment=True)
PITCH_SPEED_PROB = 0.3
PITCH_SPEED_RANGE = (0.9, 1.1)
 
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
 
class MUSDBDataset(Dataset):
    def __init__(
        self,
        records,
        sources=TARGET_SOURCES,
        segment_seconds=SEGMENT_SECONDS,
        sample_rate=SAMPLE_RATE,
        random_crop=True,
        crops_per_track=1,
        augment=True,
    ):
        self.records = records[["track_name", "path"]].to_dict("records")
        self.sources = tuple(sources)
        self.sample_rate = sample_rate
        self.segment_length = int(segment_seconds * sample_rate)
        self.random_crop = random_crop
        self.crops_per_track = max(1, int(crops_per_track))
        self.augment = augment
 
    def __len__(self):
        return len(self.records) * self.crops_per_track
 
    def _read_segment(self, audio_path, start):
        audio, sample_rate = sf.read(
            str(audio_path),
            start=start,
            frames=self.segment_length,
            dtype="float32",
            always_2d=True,
        )
        if len(audio) < self.segment_length:
            audio = np.pad(
                audio,
                ((0, self.segment_length - len(audio)), (0, 0)),
                mode="constant",
            )
        return torch.from_numpy(np.ascontiguousarray(audio.T))
 
    def _apply_pitch_speed(self, targets):
        """
        targets: [sources, channels, T]. Áp dụng CHUNG 1 speed_factor và
        1 crop_start cho toàn bộ stem để giữ đúng quan hệ thời gian giữa
        chúng (nếu để mỗi stem tự chọn crop_start riêng sẽ bị lệch pha).
        """
        speed_factor = random.uniform(*PITCH_SPEED_RANGE)
        new_len = max(int(self.segment_length / speed_factor), 1)
        if new_len >= self.segment_length:
            crop_start = random.randint(0, new_len - self.segment_length)
        else:
            crop_start = 0
 
        perturbed = torch.stack([
            pitch_speed_perturb.pitch_speed_perturb(
                targets[i], speed_factor, self.segment_length, crop_start
            )
            for i in range(targets.shape[0])
        ])
        return perturbed
 
    def __getitem__(self, idx):
        record_idx = idx // self.crops_per_track
        crop_idx = idx % self.crops_per_track
        record = self.records[record_idx]
        track_dir = Path(record["path"])
        mixture_path = track_dir / "mixture.wav"
        total_frames = sf.info(str(mixture_path)).frames
        max_start = max(0, total_frames - self.segment_length)
        if self.random_crop:
            start = np.random.randint(0, max_start + 1)
        else:
            start = int(round(crop_idx * max_start / max(1, self.crops_per_track - 1)))
 
        targets = torch.stack([
            self._read_segment(track_dir / f"{source}.wav", start)
            for source in self.sources
        ])
 
        if self.augment:
            # Pitch/speed perturbation TRƯỚC gain augmentation
            if random.random() < PITCH_SPEED_PROB:
                targets = self._apply_pitch_speed(targets)
 
            gains_db = torch.empty(len(self.sources)).uniform_(-3.0, 3.0)
            gains = torch.pow(10.0, gains_db / 20.0).view(-1, 1, 1)
            targets = targets * gains
            mixture = targets.sum(dim=0)
            peak = mixture.abs().max().clamp_min(1.0)
            targets = targets / peak
            mixture = mixture / peak
        else:
            mixture = self._read_segment(mixture_path, start)
        return mixture, targets
 
manifest = pd.read_csv("data/manifest.csv")
train_records = manifest.loc[manifest["subset"] == "train"].reset_index(drop=True)
val_records = manifest.loc[manifest["subset"] == "val"].reset_index(drop=True)
test_records = manifest.loc[manifest["subset"] == "test"].reset_index(drop=True)
 
train_dataset = MUSDBDataset(train_records, random_crop=True, crops_per_track=TRAIN_CROPS_PER_TRACK, augment=True)
# FIX: thêm augment=False rõ ràng cho val/test (bản gốc thiếu, dùng nhầm mặc định augment=True)
val_dataset = MUSDBDataset(val_records, random_crop=False, crops_per_track=EVAL_CROPS_PER_TRACK, augment=False)
test_dataset = MUSDBDataset(test_records, random_crop=False, crops_per_track=EVAL_CROPS_PER_TRACK, augment=False)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
 
print(f"Đã nạp {len(train_dataset)} mẫu train, {len(val_dataset)} mẫu val, {len(test_dataset)} mẫu test.")

Đã nạp 680 mẫu train, 120 mẫu val, 400 mẫu test.


## 6. Định nghĩa Loss & Tiện ích đánh giá

In [4]:
import math
import torch
import torch.nn.functional as F
 
EPS = 1e-8
DEFAULT_LOSS_WEIGHTS = {
    "consistency_weight": 0.10,
    "spectral_weight": 0.08,     # tăng từ 0.05
    "leakage_weight": 0.10,      # tăng từ 0.02 - thay đổi quan trọng nhất
    "si_sdr_weight": 0.01,       # MỚI - bắt đầu nhỏ, tăng dần nếu cần
}
_WINDOW_CACHE = {}
 
def _hann_window(n_fft, device):
    key = (n_fft, str(device))
    if key not in _WINDOW_CACHE:
        _WINDOW_CACHE[key] = torch.hann_window(n_fft, device=device, dtype=torch.float32)
    return _WINDOW_CACHE[key]
 
def multi_resolution_stft_loss(estimate, target, fft_sizes=(256, 512, 1024), eps=EPS):
    # THAY ĐỔI: thêm 256 vào fft_sizes để nhạy hơn với lỗi tần số cao
    estimate = estimate.float().mean(dim=-2).reshape(-1, estimate.shape[-1])
    target = target.float().mean(dim=-2).reshape(-1, target.shape[-1])
    total = estimate.new_tensor(0.0)
    for n_fft in fft_sizes:
        hop = n_fft // 4
        window = _hann_window(n_fft, estimate.device)
        est_mag = torch.stft(estimate, n_fft, hop, window=window, return_complex=True).abs()
        tgt_mag = torch.stft(target, n_fft, hop, window=window, return_complex=True).abs()
        spectral_convergence = (est_mag - tgt_mag).norm() / (tgt_mag.norm() + eps)
        log_magnitude = F.l1_loss(torch.log1p(est_mag), torch.log1p(tgt_mag))
        total = total + spectral_convergence + log_magnitude
    return total / len(fft_sizes)
 
def leakage_residual_loss(estimate, target, eps=EPS):
    # KHÔNG đổi logic - chỉ tăng trọng số khi gọi (leakage_weight) ở separation_loss
    residual = (estimate - target).float().flatten(start_dim=2)
    nuisance = target.float().flatten(start_dim=2)
    dot = torch.einsum("bin,bjn->bij", residual, nuisance)
    denom = (residual.square().sum(-1)[:, :, None] * nuisance.square().sum(-1)[:, None, :])
    corr_squared = dot.square() / (denom + eps)
    mask = ~torch.eye(corr_squared.shape[-1], dtype=torch.bool, device=corr_squared.device)
    return corr_squared[:, mask].mean()
 
def leakage_db(estimate, target, eps=EPS):
    value = leakage_residual_loss(estimate, target, eps=eps)
    return 10.0 * torch.log10(value + eps)
 
def si_sdr_loss(estimate, target, eps=EPS):
    """
    MỚI: Scale-Invariant SDR loss.
    Phạt trực tiếp việc estimate bị lệch BIÊN ĐỘ so với target (dù đúng hướng/
    dạng sóng), điều mà L1 loss thường bỏ qua vì L1 trung bình vẫn nhỏ ngay cả
    khi biên độ sai lệch ~50% (đúng như vocals đang bị "hát nhỏ đi").
 
    estimate, target: [B, sources, channels, T]
    Trả về: loss cần MINIMIZE (âm của SI-SDR trung bình).
    """
    b, s, c, t = estimate.shape
    est = estimate.float().reshape(b * s * c, t)
    tgt = target.float().reshape(b * s * c, t)
 
    # Zero-mean theo thời gian (chuẩn SI-SDR)
    est = est - est.mean(dim=-1, keepdim=True)
    tgt = tgt - tgt.mean(dim=-1, keepdim=True)
 
    dot = (est * tgt).sum(dim=-1, keepdim=True)
    tgt_energy = tgt.pow(2).sum(dim=-1, keepdim=True) + eps
    proj = dot / tgt_energy * tgt  # hình chiếu estimate lên target (thành phần "đúng hướng")
    noise = est - proj             # phần còn lại (nhiễu/lệch scale)
 
    si_sdr_val = 10.0 * torch.log10(
        (proj.pow(2).sum(dim=-1) + eps) / (noise.pow(2).sum(dim=-1) + eps)
    )
    return -si_sdr_val.mean()  # minimize -> maximize SI-SDR thật
 
def separation_loss(
    estimate, target, mixture,
    consistency_weight=0.10, spectral_weight=0.08, leakage_weight=0.10, si_sdr_weight=0.01,
):
    source_l1 = F.l1_loss(estimate, target)
    mixture_l1 = F.l1_loss(estimate.sum(dim=1), mixture)
    spectral = multi_resolution_stft_loss(estimate, target) if spectral_weight > 0 else source_l1.new_zeros(())
    leakage = leakage_residual_loss(estimate, target) if leakage_weight > 0 else source_l1.new_zeros(())
    si_sdr = si_sdr_loss(estimate, target) if si_sdr_weight > 0 else source_l1.new_zeros(())
 
    total = (
        source_l1
        + consistency_weight * mixture_l1
        + spectral_weight * spectral
        + leakage_weight * leakage
        + si_sdr_weight * si_sdr
    )
    return total, {
        "source_l1": source_l1.detach(),
        "mixture_l1": mixture_l1.detach(),
        "spectral": spectral.detach(),
        "leakage": leakage.detach(),
        "si_sdr": si_sdr.detach(),  # theo dõi riêng để biết có cần tăng si_sdr_weight không
    }
 
def sdr_per_source_db(estimate, target, eps=EPS):
    target_energy = target.pow(2).sum(dim=(-2, -1))
    error_energy = (target - estimate).pow(2).sum(dim=(-2, -1))
    return 10.0 * torch.log10((target_energy + eps) / (error_energy + eps))
 
def sdr_db(estimate, target, eps=EPS):
    return sdr_per_source_db(estimate, target, eps=eps).mean()

## 7. Huấn luyện và lưu Checkpoint cục bộ

In [ ]:
import time
from contextlib import nullcontext
import torch.optim as optim
from tqdm.auto import tqdm
from scripts.demucs_lite import DemucsLite

EPOCHS = 100
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
MODEL_CHANNELS = 32  # Giữ nguyên cấu trúc gốc
MODEL_DEPTH = 5     # Giữ nguyên cấu trúc gốc
RESUME = True
REMIX_PROB = 0.5  # xác suất 1 track trong batch bị remix (thay vì giữ nguyên bộ 4 stem gốc)

SAVE_DIR = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
LAST_CHECKPOINT = os.path.join(SAVE_DIR, "demucs_lite_last.pth")
BEST_CHECKPOINT = os.path.join(SAVE_DIR, "demucs_lite_best.pth")
HISTORY_PATH = os.path.join(SAVE_DIR, "training_history.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")
 
def remix_batch(mixture, target, remix_prob=REMIX_PROB):
    """
    target: [B, sources, channels, T]
    mixture: [B, channels, T] (sẽ được tính lại, giá trị đầu vào bị bỏ qua)
 
    Với mỗi track trong batch, với xác suất remix_prob: với MỖI source,
    độc lập chọn ngẫu nhiên lấy source đó từ 1 track KHÁC trong cùng batch
    (hoán vị ngẫu nhiên theo batch dimension cho từng source riêng biệt).
    Sau đó tính lại mixture = tổng các source đã remix.
    """
    B, S, C, T = target.shape
    new_target = target.clone()
 
    for s in range(S):
        if torch.rand(1).item() < remix_prob:
            perm = torch.randperm(B, device=target.device)
            new_target[:, s] = target[perm, s]
 
    new_mixture = new_target.sum(dim=1)  # [B, channels, T]
    return new_mixture, new_target
 
def amp_context():
    return torch.amp.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext()

train_model = DemucsLite(
    sources=len(TARGET_SOURCES),
    audio_channels=2,
    channels=MODEL_CHANNELS,
    depth=MODEL_DEPTH,
).to(device)

optimizer = optim.AdamW(train_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_items = 0
    progress = tqdm(loader, desc="Train", leave=False)

    for mixture, target in progress:
        mixture = mixture.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        mixture, target = remix_batch(mixture, target)
        optimizer.zero_grad(set_to_none=True)

        with amp_context():
            estimate = model(mixture)
            loss, _ = separation_loss(estimate, target, mixture, **DEFAULT_LOSS_WEIGHTS)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        batch_size = mixture.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size
        progress.set_postfix(loss=f"{loss.item():.4f}", grad=f"{float(grad_norm):.2f}")

    return total_loss / max(total_items, 1)

@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0.0
    total_sdr_by_source = torch.zeros(len(TARGET_SOURCES), dtype=torch.float64)
    total_items = 0

    for mixture, target in tqdm(loader, desc="Validation", leave=False):
        mixture = mixture.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        with amp_context():
            estimate = model(mixture)
            loss, _ = separation_loss(estimate, target, mixture, **DEFAULT_LOSS_WEIGHTS)

        batch_size = mixture.size(0)
        total_loss += loss.item() * batch_size
        # batch_sdr: [batch, sources] -> giữ nguyên chiều "sources" khi cộng dồn,
        # KHÔNG mean() ngay để không mất thông tin SDR riêng từng stem
        batch_sdr = sdr_per_source_db(estimate.float(), target.float()).detach().cpu()
        total_sdr_by_source += batch_sdr.sum(dim=0).double()
        total_items += batch_size

    denominator = max(total_items, 1)
    sdr_by_source = total_sdr_by_source / denominator  # tensor [sources], 1 giá trị dB / stem

    # Dict SDR riêng từng nguồn, khớp thứ tự TARGET_SOURCES = ("vocals","drums","bass","other")
    sdr_per_source = {
        f"sdr_{name}_db": sdr_by_source[i].item()
        for i, name in enumerate(TARGET_SOURCES)
    }

    return {
        "loss": total_loss / denominator,
        "sdr_db": sdr_by_source.mean().item(),  # trung bình toàn bộ (giữ lại để so sánh xu hướng chung)
        **sdr_per_source,
    }

start_epoch = 1
best_val_loss = float("inf")
history = []

if RESUME and os.path.isfile(LAST_CHECKPOINT):
    checkpoint = torch.load(LAST_CHECKPOINT, map_location=device, weights_only=False)
    train_model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint.get("best_val_loss", float("inf"))
    history = checkpoint.get("history", [])
    print(f"Tiếp tục huấn luyện từ epoch {checkpoint['epoch']}.")

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_loss = train_one_epoch(train_model, train_loader)
    val_metrics = validate(train_model, val_loader)
    scheduler.step(val_metrics["loss"])
    epoch_time_sec = time.perf_counter() - epoch_start

    is_best = val_metrics["loss"] < best_val_loss
    if is_best:
        best_val_loss = val_metrics["loss"]

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": train_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
    }
    torch.save(checkpoint, LAST_CHECKPOINT)
    if is_best:
        torch.save(checkpoint, BEST_CHECKPOINT)

    # Ghi đầy đủ SDR từng stem vào history (mỗi dòng CSV có sdr_vocals_db, sdr_drums_db, sdr_bass_db, sdr_other_db)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_sdr_db": val_metrics["sdr_db"],
        "sdr_vocals_db": val_metrics["sdr_vocals_db"],
        "sdr_drums_db": val_metrics["sdr_drums_db"],
        "sdr_bass_db": val_metrics["sdr_bass_db"],
        "sdr_other_db": val_metrics["sdr_other_db"],
        "time_sec": epoch_time_sec,
    })
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    # In ra console SDR riêng từng stem mỗi epoch, dễ theo dõi stem nào yếu nhất
    per_source_str = " | ".join(
        f"{name}: {val_metrics[f'sdr_{name}_db']:.2f}dB" for name in TARGET_SOURCES
    )
    print(
        f"Epoch {epoch:03d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} "
        f"| Val SDR (avg): {val_metrics['sdr_db']:.2f} dB | {per_source_str} | {epoch_time_sec:.1f}s"
    )

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tiếp tục huấn luyện từ epoch 77.


Epoch 078/100 | Train Loss: 0.1215 | Val Loss: 0.1929 | Val SDR (avg): -7.81 dB | vocals: -19.35dB | drums: -4.48dB | bass: -7.22dB | other: -0.20dB | 1783.8s


Epoch 079/100 | Train Loss: 0.1218 | Val Loss: 0.1922 | Val SDR (avg): -7.78 dB | vocals: -19.38dB | drums: -4.45dB | bass: -7.12dB | other: -0.16dB | 1650.5s


Epoch 080/100 | Train Loss: 0.1182 | Val Loss: 0.1911 | Val SDR (avg): -7.69 dB | vocals: -19.19dB | drums: -4.29dB | bass: -7.11dB | other: -0.15dB | 1671.1s


...

Epoch 084/100 | Train Loss: 0.1170 | Val Loss: 0.1903 | Val SDR (avg): -7.66 dB | vocals: -19.21dB | drums: -4.34dB | bass: -7.02dB | other: -0.08dB | 1527.7s


Epoch 085/100 | Train Loss: 0.1184 | Val Loss: 0.1901 | Val SDR (avg): -7.65 dB | vocals: -19.18dB | drums: -4.34dB | bass: -6.99dB | other: -0.07dB | 1522.5s


Epoch 086/100 | Train Loss: 0.1314 | Val Loss: 0.1898 | Val SDR (avg): -7.65 dB | vocals: -19.22dB | drums: -4.31dB | bass: -6.99dB | other: -0.07dB | 1526.9s


Epoch 087/100 | Train Loss: 0.1272 | Val Loss: 0.1898 | Val SDR (avg): -7.65 dB | vocals: -19.22dB | drums: -4.33dB | bass: -6.97dB | other: -0.08dB | 1534.1s


Train:  86%|████████▋ | 147/170 [22:56<06:55, 18.05s/it, grad=0.11, loss=0.1187]

## 8. Tinh chỉnh Hyperparameters (Đã sửa lỗi autograd RuntimeError)

In [ ]:
import gc

RUN_HYPERPARAMETER_SEARCH = True
TUNE_EPOCHS = 1
TUNE_MAX_TRAIN_BATCHES = 10
TUNE_MAX_VAL_BATCHES = 5

HYPERPARAMETER_CANDIDATES = [
    {"name": "fast", "channels": 24, "depth": 4, "lr": 5e-4, "consistency_weight": 0.10, "spectral_weight": 0.03, "leakage_weight": 0.01},
    {"name": "balanced", "channels": 32, "depth": 5, "lr": 3e-4, "consistency_weight": 0.10, "spectral_weight": 0.05, "leakage_weight": 0.02},
]

# Batch giả lập độc lập để đo RTF an toàn
dummy_bench = torch.randn(1, 2, int(3 * SAMPLE_RATE), device=device)

@torch.no_grad()
def benchmark_speed(model, sample_input, repeats=5):
    model.eval()
    latencies = []
    for _ in range(repeats):
        start = time.perf_counter()
        with amp_context():
            _ = model(sample_input)
        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - start)
    return float(np.mean(latencies))

def train_short_trial(model, optimizer, scaler, config):
    model.train()
    running_loss = 0.0
    item_count = 0
    for batch_idx, (mixture, target) in enumerate(train_loader):
        if batch_idx >= TUNE_MAX_TRAIN_BATCHES:
            break
        # Tách biệt tensor khỏi buffer bằng .clone()
        mixture = mixture.to(device, non_blocking=True).clone()
        target = target.to(device, non_blocking=True).clone()
        
        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            estimate = model(mixture)
            loss, _ = separation_loss(
                estimate, target, mixture,
                consistency_weight=config["consistency_weight"],
                spectral_weight=config["spectral_weight"],
                leakage_weight=config["leakage_weight"]
            )
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * mixture.shape[0]
        item_count += mixture.shape[0]
    return running_loss / max(item_count, 1)

@torch.no_grad()
def validate_short_trial(model, config):
    model.eval()
    loss_sum = 0.0
    item_count = 0
    for batch_idx, (mixture, target) in enumerate(val_loader):
        if batch_idx >= TUNE_MAX_VAL_BATCHES:
            break
        mixture = mixture.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        with amp_context():
            estimate = model(mixture)
            loss, _ = separation_loss(
                estimate, target, mixture,
                consistency_weight=config["consistency_weight"],
                spectral_weight=config["spectral_weight"],
                leakage_weight=config["leakage_weight"]
            )
        loss_sum += loss.item() * mixture.shape[0]
        item_count += mixture.shape[0]
    return {"val_loss": loss_sum / max(item_count, 1)}

def run_trial(config):
    model = DemucsLite(
        sources=len(TARGET_SOURCES), audio_channels=2,
        channels=config["channels"], depth=config["depth"],
    ).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=WEIGHT_DECAY)
    trial_scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    
    for _ in range(TUNE_EPOCHS):
        last_loss = train_short_trial(model, optimizer, trial_scaler, config)
    
    val_res = validate_short_trial(model, config)
    lat = benchmark_speed(model, dummy_bench, repeats=5)
    rtf = lat / (dummy_bench.shape[-1] / SAMPLE_RATE)
    
    del model, optimizer, trial_scaler
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {**config, "train_loss": last_loss, **val_res, "latency_ms": lat * 1000, "rtf": rtf}

if RUN_HYPERPARAMETER_SEARCH:
    results = [run_trial(cfg) for cfg in HYPERPARAMETER_CANDIDATES]
    df_tune = pd.DataFrame(results)
    print("\nKết quả thử nghiệm:")
    print(df_tune[["name", "channels", "depth", "train_loss", "val_loss", "latency_ms", "rtf"]])


Kết quả thử nghiệm:
       name  channels  depth  train_loss  val_loss  latency_ms       rtf
0      fast        24      4    0.130467  0.119016   151.61490  0.050538
1  balanced        32      5    0.170702  0.187649   214.27088  0.071424


In [ ]:
"""
CELL MỚI - thêm vào CUỐI notebook, chạy SAU khi cell 2, 4, 9, 11, 13 (gốc) đã
chạy ít nhất 1 lần trong kernel hiện tại. KHÔNG cần sửa bất kỳ cell nào có sẵn.

Cell này tự chứa toàn bộ phần cần thiết: định nghĩa SDR loại trừ đoạn im lặng
+ load checkpoint + evaluate. Không train, không đụng tới train_model/optimizer
đang có trong bộ nhớ (nếu có).
"""
import torch
from tqdm.auto import tqdm
from contextlib import nullcontext
from scripts.demucs_lite import DemucsLite

EPS = 1e-8

# --- 1. Hàm SDR loại trừ đoạn target gần như im lặng ---
def sdr_per_source_db_masked(estimate, target, eps=EPS, silence_threshold_db=-40.0):
    target_energy = target.pow(2).sum(dim=(-2, -1))
    error_energy = (target - estimate).pow(2).sum(dim=(-2, -1))

    n_samples = target.shape[-1] * target.shape[-2]
    silence_energy_threshold = n_samples * (10.0 ** (silence_threshold_db / 10.0))

    active_mask = target_energy > silence_energy_threshold
    sdr_values = 10.0 * torch.log10((target_energy + eps) / (error_energy + eps))
    sdr_values = torch.where(active_mask, sdr_values, torch.full_like(sdr_values, float("nan")))
    return sdr_values, active_mask


# --- 2. Load checkpoint đã train (không train tiếp) ---
CHECKPOINT_TO_EVAL = "checkpoints/demucs_lite_best.pth"  # đổi sang _last.pth nếu muốn eval bản mới nhất

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")

def amp_context():
    return torch.amp.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext()

eval_model = DemucsLite(
    sources=len(TARGET_SOURCES),
    audio_channels=2,
    channels=32,   # phải khớp MODEL_CHANNELS lúc train (cell 15)
    depth=5,       # phải khớp MODEL_DEPTH lúc train (cell 15)
).to(device)

ckpt = torch.load(CHECKPOINT_TO_EVAL, map_location=device, weights_only=False)
eval_model.load_state_dict(ckpt["model_state_dict"])
eval_model.eval()
print(f"Đã load checkpoint epoch {ckpt.get('epoch', '?')} từ {CHECKPOINT_TO_EVAL}")


# --- 3. Evaluate trên val_loader (đã có sẵn từ cell 11) ---
@torch.no_grad()
def evaluate_masked(model, loader):
    sdr_sum = torch.zeros(len(TARGET_SOURCES), dtype=torch.float64)
    active_count = torch.zeros(len(TARGET_SOURCES), dtype=torch.float64)
    total_entries = torch.zeros(len(TARGET_SOURCES), dtype=torch.float64)

    for mixture, target in tqdm(loader, desc="Evaluate (masked SDR)"):
        mixture = mixture.to(device)
        target = target.to(device)
        with amp_context():
            estimate = model(mixture)

        sdr_values, active_mask = sdr_per_source_db_masked(estimate.float(), target.float())
        sdr_values = sdr_values.detach().cpu()
        active_mask = active_mask.detach().cpu()

        sdr_sum += torch.nan_to_num(sdr_values, nan=0.0).sum(dim=0).double()
        active_count += active_mask.sum(dim=0).double()
        total_entries += mixture.size(0)

    denom = active_count.clamp(min=1)
    sdr_by_source = sdr_sum / denom
    active_ratio = active_count / total_entries.clamp(min=1)

    print("\n=== KẾT QUẢ SDR (đã loại trừ đoạn im lặng) ===")
    for i, name in enumerate(TARGET_SOURCES):
        print(
            f"{name:8s}: SDR = {sdr_by_source[i]:6.2f} dB "
            f"(tính trên {active_ratio[i]*100:.1f}% đoạn có tín hiệu)"
        )
    print(f"{'Trung bình':8s}: SDR = {sdr_by_source.mean():6.2f} dB")
    return sdr_by_source, active_ratio

sdr_by_source, active_ratio = evaluate_masked(eval_model, val_loader)

Đã load checkpoint epoch 76 từ checkpoints/demucs_lite_best.pth


Evaluate (masked SDR): 100%|██████████| 30/30 [01:07<00:00,  2.25s/it]


=== KẾT QUẢ SDR (đã loại trừ đoạn im lặng) ===
vocals  : SDR =   2.65 dB (tính trên 71.7% đoạn có tín hiệu)
drums   : SDR =   3.66 dB (tính trên 80.8% đoạn có tín hiệu)
bass    : SDR =   3.15 dB (tính trên 80.8% đoạn có tín hiệu)
other   : SDR =   2.79 dB (tính trên 93.3% đoạn có tín hiệu)
Trung bình: SDR =   3.06 dB


In [ ]:
# import librosa
# import librosa.display
# import matplotlib.pyplot as plt
# import numpy as np

# # Đường dẫn 2 file cần so sánh
# CLEAN_FILE = "path/to/reference_clean.wav"      # File mẫu chuẩn
# ESTIMATED_FILE = "path/to/separated_audio.wav" # File sau khi tách

# # 1. Load âm thanh (đồng bộ sample rate)
# y_clean, sr = librosa.load(CLEAN_FILE, sr=None)
# y_est, _ = librosa.load(ESTIMATED_FILE, sr=sr)

# # Đồng bộ độ dài 2 mảng nếu có chênh lệch nhỏ
# min_len = min(len(y_clean), len(y_est))
# y_clean, y_est = y_clean[:min_len], y_est[:min_len]

# # 2. Tính Spectrogram (DB scale)
# D_clean = librosa.amplitude_to_db(np.abs(librosa.stft(y_clean)), ref=np.max)
# D_est = librosa.amplitude_to_db(np.abs(librosa.stft(y_est)), ref=np.max)

# # 3. Vẽ biểu đồ so sánh trực quan
# fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

# # Waveform (Dạng sóng)
# librosa.display.waveshow(y_clean, sr=sr, ax=axes[0, 0], color='blue')
# axes[0, 0].set_title("Waveform: File Mẫu (Reference/Ground Truth)")
# axes[0, 0].set_ylabel("Amplitude")

# librosa.display.waveshow(y_est, sr=sr, ax=axes[0, 1], color='orange')
# axes[0, 1].set_title("Waveform: File Tách Được (Separated)")

# # Spectrogram (Phổ tần số)
# img1 = librosa.display.specshow(D_clean, sr=sr, x_axis='time', y_axis='hz', ax=axes[1, 0], cmap='magma')
# axes[1, 0].set_title("Spectrogram: File Mẫu")
# fig.colorbar(img1, ax=axes[1, 0], format="%+2.0f dB")

# img2 = librosa.display.specshow(D_est, sr=sr, x_axis='time', y_axis='hz', ax=axes[1, 1], cmap='magma')
# axes[1, 1].set_title("Spectrogram: File Tách Được")
# fig.colorbar(img2, ax=axes[1, 1], format="%+2.0f dB")

# plt.tight_layout()
# plt.show()